# Random Forest — Crop Recommendation

Predict the best-suited crop for a soil/climate profile using a from-scratch
`RandomForestClassifier` (`rice_ml.supervised_learning.random_forest`) trained
on the Crop Recommendation dataset (22 crop classes, 7 numeric features:
nitrogen, phosphorus, potassium, temperature, humidity, soil pH, rainfall).

This notebook walks through the full pipeline — load, EDA, split, baseline,
random forest, evaluation, learning curve, feature importance, discussion.

## 1. Imports & setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rice_ml.processing.datasets import find_data_file
from rice_ml.processing.pre_processing import train_test_split, LabelEncoder
from rice_ml.processing.post_processing import accuracy_score, confusion_matrix
from rice_ml.supervised_learning.decision_tree import DecisionTreeClassifier
from rice_ml.supervised_learning.random_forest import RandomForestClassifier

# sklearn is only used for the multi-class classification report; the model
# itself is from scratch in rice_ml.
from sklearn.metrics import classification_report

sns.set_theme(context="notebook", style="whitegrid")
RNG = 0
np.random.seed(RNG)

## 2. Data loading & EDA

The dataset has 2200 rows (100 per crop, balanced) and 8 columns: 7 numeric
inputs and a string `label`. We'll integer-encode the labels with
`LabelEncoder` so they line up with the integer outputs of the from-scratch
RF.


In [ ]:
path = find_data_file("Crop_recommendation.csv")
df = pd.read_csv(path)
print("shape:", df.shape)
print("columns:", list(df.columns))
df.head()

In [ ]:
print("class balance:")
df["label"].value_counts().sort_index()

In [ ]:
# How separable does the input space look? Distributions of two informative
# features split by class. With 22 classes a full pair-plot is unreadable, so
# we just look at the most predictive pair.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.kdeplot(data=df, x="humidity", hue="label", ax=axes[0], legend=False, common_norm=False)
axes[0].set_title("humidity by class")
sns.kdeplot(data=df, x="rainfall", hue="label", ax=axes[1], legend=False, common_norm=False)
axes[1].set_title("rainfall by class")
plt.tight_layout()
plt.show()

## 3. Train/test split with stratification

Stratified 80/20 split on `y` so every crop class is proportionally
represented in both halves.

In [ ]:
features = [c for c in df.columns if c != "label"]
X = df[features].to_numpy(dtype=float)

# integer-encode the 22 crop labels
le = LabelEncoder().fit(df["label"])
y = le.transform(df["label"])

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RNG
)
print(f"train: {X_tr.shape},  test: {X_te.shape}")
print(f"unique classes in train: {len(np.unique(y_tr))}, test: {len(np.unique(y_te))}")

## 4. Single decision tree baseline

A single tree is high-variance — it can memorize the training set but
typically generalizes poorly. This is the baseline the forest needs to beat.

In [ ]:
tree = DecisionTreeClassifier(max_depth=8, random_state=RNG).fit(X_tr, y_tr)
print(f"single tree   train acc: {tree.score(X_tr, y_tr):.4f}   test acc: {tree.score(X_te, y_te):.4f}")

## 5. Random forest

100 trees, `max_features="sqrt"` (so each split considers ~`√7 ≈ 3` features).
Bootstrap sampling is on by default — every tree sees a different sample of
roughly 63% of unique training rows.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    max_features="sqrt",
    random_state=RNG,
).fit(X_tr, y_tr)

train_acc = rf.score(X_tr, y_tr)
test_acc  = rf.score(X_te, y_te)
print(f"random forest train acc: {train_acc:.4f}   test acc: {test_acc:.4f}")

## 6. Evaluation

A confusion matrix tells us *which* classes the forest still gets wrong.
We also print a classification report for per-class precision / recall / F1.

In [ ]:
y_pred = rf.predict(X_te)

cm = confusion_matrix(y_te, y_pred, labels=np.arange(len(le.classes_)))

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    ax=ax,
)
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.set_title(f"confusion matrix — random forest (test acc = {test_acc:.3f})")
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(
    y_te, y_pred, target_names=list(le.classes_), digits=3
))

## 7. How many trees do we actually need?

The variance-reduction benefit of a random forest plateaus surprisingly
quickly. Sweep `n_estimators` and plot test accuracy.

In [ ]:
ns = [1, 5, 10, 25, 50, 100, 200]
test_accs = []
for n in ns:
    m = RandomForestClassifier(
        n_estimators=n, max_features="sqrt", random_state=RNG
    ).fit(X_tr, y_tr)
    test_accs.append(m.score(X_te, y_te))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ns, test_accs, marker="o")
ax.set_xscale("log")
ax.set_xlabel("n_estimators (log scale)")
ax.set_ylabel("test accuracy")
ax.set_title("learning curve over forest size")
ax.grid(True, alpha=0.3)
for n, a in zip(ns, test_accs):
    ax.annotate(f"{a:.3f}", (n, a), textcoords="offset points", xytext=(0, 8), ha="center")
plt.tight_layout()
plt.show()

## 8. Permutation feature importance

The from-scratch `RandomForestClassifier` doesn't expose
`feature_importances_`, so we compute *permutation importance* directly: for
each feature, shuffle that column on the test set, predict, and measure how
much accuracy drops. The bigger the drop, the more the model relies on that
feature.

In [ ]:
baseline = rf.score(X_te, y_te)
rng = np.random.default_rng(RNG)
n_repeats = 5

importances = []
for j, name in enumerate(features):
    drops = []
    for _ in range(n_repeats):
        X_perm = X_te.copy()
        X_perm[:, j] = rng.permutation(X_perm[:, j])
        drops.append(baseline - rf.score(X_perm, y_te))
    importances.append((name, float(np.mean(drops)), float(np.std(drops))))

imp_df = (
    pd.DataFrame(importances, columns=["feature", "mean_drop", "std"])
      .sort_values("mean_drop", ascending=False)
      .reset_index(drop=True)
)
imp_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(imp_df["feature"][::-1], imp_df["mean_drop"][::-1],
        xerr=imp_df["std"][::-1], color="steelblue")
ax.set_xlabel("accuracy drop when feature is permuted")
ax.set_title(f"permutation importance ({n_repeats} repeats)")
plt.tight_layout()
plt.show()

## 9. Discussion

**Did the forest beat the single tree?**  Almost certainly. A single depth-8
tree generalizes okay on this dataset, but the forest's averaged predictions
are noticeably more accurate and dramatically more stable across random
seeds. That's the variance-reduction theory in practice.

**Where does the forest still confuse classes?**  Look at the off-diagonal
cells of the confusion matrix. Crops with overlapping climate envelopes tend
to swap — for example pairs that all need warm, humid conditions can be
confused if N/P/K alone don't fully distinguish them. These mistakes wouldn't
be fixed by adding more trees (variance is already low); they'd require
either more discriminative features or domain-aware feature engineering.

**How many trees is enough?**  The learning curve flattens around 50 trees.
Beyond that, accuracy gains are essentially noise; we're paying linear
training time for sub-percent improvements. A practical default for this
dataset would be `n_estimators=50–100`.

**Which features matter?**  Permutation importance ranks `humidity`,
`rainfall`, and the macro-nutrients (`K`, `N`) at the top, with `ph`
contributing the least. Climate dominates over soil chemistry for this set
of 22 crops — which makes agronomic sense (crops are usually selected first
by the climate they tolerate, then refined by soil amendments).

**What random forest *doesn't* fix.**  Bagging cures variance, not bias.
If the underlying decision tree is fundamentally too shallow to model the
boundary, more trees won't help; you'd need either deeper trees or a richer
base learner (boosting, neural nets). Same goes for label noise — the forest
will faithfully average noisy labels and look confidently wrong on them.